In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
img_size = 112
channels = 3
num_classes = 2
latent_dim = 1000
batch_size = 32

Original generator 1

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        self.label_emb = nn.Embedding(num_classes, 1)

        self.init_size = img_size // 16  # 7
        self.l1 = nn.Sequential(nn.Linear(latent_dim, 1024 * self.init_size * self.init_size))

        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(1025),

            nn.ConvTranspose2d(1025, 512, 5, stride=2, padding=2, output_padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            nn.ConvTranspose2d(512, 256, 5, stride=2, padding=2, output_padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 5, stride=2, padding=2, output_padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, channels, 5, stride=2, padding=2, output_padding=1),
            nn.Tanh()
        )

    def forward(self, noise, labels):
        out = self.l1(noise)
        out = out.view(out.shape[0], 1024, 7, 7)

        label = self.label_emb(labels).view(labels.size(0), 1, 1, 1)
        label = label.repeat(1, 1, 7, 7)

        out = torch.cat((out, label), dim=1)
        img = self.conv_blocks(out)
        return img

Modified generator: ResBlock added, Conv->Unsample

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

img_size = 112
channels = 3
num_classes = 2
latent_dim = 256


class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + residual)


class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        self.label_emb = nn.Embedding(num_classes, 16)

        self.init_size = img_size // 16  # 7
        self.fc = nn.Linear(latent_dim + 16, 256 * self.init_size * self.init_size)

        self.block1 = nn.Sequential(
            nn.BatchNorm2d(256),
            ResBlock(256)
        )

        self.block2 = nn.Sequential(
            nn.Upsample(scale_factor=2),  # 7→14
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            ResBlock(256)
        )

        self.block3 = nn.Sequential(
            nn.Upsample(scale_factor=2),  # 14→28
            nn.Conv2d(256, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            ResBlock(128)
        )

        self.block4 = nn.Sequential(
            nn.Upsample(scale_factor=2),  # 28→56
            nn.Conv2d(128, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            ResBlock(64)
        )

        self.block5 = nn.Sequential(
            nn.Upsample(scale_factor=2),  # 56→112
            nn.Conv2d(64, channels, 3, padding=1),
            nn.Tanh()
        )

    def forward(self, noise, labels):
        label_embed = self.label_emb(labels)

        x = torch.cat([noise, label_embed], dim=1)

        out = self.fc(x)
        out = out.view(out.size(0), 256, self.init_size, self.init_size)

        out = self.block1(out)
        out = self.block2(out)
        out = self.block3(out)
        out = self.block4(out)
        img = self.block5(out)

        return img

In [4]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        def block(in_feat, out_feat, stride):
            return [
                nn.Conv2d(in_feat, out_feat, 3, stride=stride, padding=1),
                nn.BatchNorm2d(out_feat),
                nn.LeakyReLU(0.2, inplace=True),
                nn.Dropout(0.5)
            ]

        self.conv = nn.Sequential(
            *block(channels, 32, 1),
            *block(32, 64, 2),
            *block(64, 128, 2),
            *block(128, 256, 2),
            *block(256, 512, 2),
        )

        ds_size = img_size // 16
        self.flatten = nn.Flatten()

        self.adv_layer = nn.Linear(512 * ds_size * ds_size, 1)
        self.aux_layer = nn.Linear(512 * ds_size * ds_size, num_classes)

    def forward(self, img):
        out = self.conv(img)
        out = self.flatten(out)

        validity = torch.sigmoid(self.adv_layer(out))
        label = torch.softmax(self.aux_layer(out), dim=1)

        return validity, label

In [5]:
transform = transforms.Compose([
    transforms.Resize((112,112)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_dataset = datasets.ImageFolder("data/train", transform=transform)
dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [6]:
print(train_dataset.classes)

['0_Normal', '1_Covid19']


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generator = Generator().to(device)
discriminator = Discriminator().to(device)

optimizer_G = optim.Adam(generator.parameters(), lr=0.0001, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

adversarial_loss = nn.BCELoss()
auxiliary_loss = nn.CrossEntropyLoss()

In [8]:
print(device)

cuda


In [9]:
#第一个训练
for epoch in range(50):
    for i, (imgs, labels) in enumerate(dataloader):

        batch_size = imgs.size(0)

        real_imgs = imgs.to(device)
        labels = labels.to(device)

        valid = torch.ones(batch_size, 1).to(device)
        fake = torch.zeros(batch_size, 1).to(device)

        optimizer_G.zero_grad()

        noise = torch.randn(batch_size, latent_dim).to(device)

        gen_labels = torch.randint(0, num_classes, (batch_size,)).to(device)

        gen_imgs = generator(noise, gen_labels)

        validity, pred_label = discriminator(gen_imgs)

        g_loss = adversarial_loss(validity, valid) + auxiliary_loss(pred_label, gen_labels)
        g_loss.backward()
        optimizer_G.step()

        # ---------------------
        # Train Discriminator
        # ---------------------
        optimizer_D.zero_grad()

        real_pred, real_aux = discriminator(real_imgs)
        d_real_loss = adversarial_loss(real_pred, valid) + auxiliary_loss(real_aux, labels)

        fake_pred, fake_aux = discriminator(gen_imgs.detach())
        d_fake_loss = adversarial_loss(fake_pred, fake) + auxiliary_loss(fake_aux, gen_labels)

        d_loss = (d_real_loss + d_fake_loss) / 2

        d_loss.backward()
        optimizer_D.step()

    print(f"[Epoch {epoch}] D loss: {d_loss.item():.4f}, G loss: {g_loss.item():.4f}")

[Epoch 0] D loss: 0.7549, G loss: 5.1835
[Epoch 1] D loss: 0.9775, G loss: 1.7134
[Epoch 2] D loss: 1.6114, G loss: 1.8067
[Epoch 3] D loss: 1.5334, G loss: 0.7747
[Epoch 4] D loss: 1.5087, G loss: 0.5190



KeyboardInterrupt



In [ ]:
G_losses = []
D_losses = []

for epoch in range(50):

    g_epoch_loss = 0
    d_epoch_loss = 0
    count = 0

    for i, (imgs, labels) in enumerate(dataloader):

        batch_size = imgs.size(0)

        real_imgs = imgs.to(device)
        labels = labels.to(device)

        valid = torch.ones(batch_size, 1).to(device)
        fake = torch.zeros(batch_size, 1).to(device)

        optimizer_G.zero_grad()

        noise = torch.randn(batch_size, latent_dim).to(device)
        gen_labels = torch.randint(0, num_classes, (batch_size,)).to(device)

        gen_imgs = generator(noise, gen_labels)

        validity, pred_label = discriminator(gen_imgs)

        g_loss = adversarial_loss(validity, valid) + auxiliary_loss(pred_label, gen_labels)
        g_loss.backward()
        optimizer_G.step()

        optimizer_D.zero_grad()

        real_pred, real_aux = discriminator(real_imgs)
        d_real_loss = adversarial_loss(real_pred, valid) + auxiliary_loss(real_aux, labels)

        fake_pred, fake_aux = discriminator(gen_imgs.detach())
        d_fake_loss = adversarial_loss(fake_pred, fake) + auxiliary_loss(fake_aux, gen_labels)

        d_loss = (d_real_loss + d_fake_loss) / 2

        d_loss.backward()
        optimizer_D.step()

        g_epoch_loss += g_loss.item()
        d_epoch_loss += d_loss.item()
        count += 1

    avg_g = g_epoch_loss / count
    avg_d = d_epoch_loss / count

    G_losses.append(avg_g)
    D_losses.append(avg_d)

    print(f"[Epoch {epoch}] D loss: {avg_d:.4f}, G loss: {avg_g:.4f}")

In [ ]:
torch.save(generator.state_dict(), "generator_2.pth")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(G_losses, label="Generator Loss")
plt.plot(D_losses, label="Discriminator Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()
plt.grid()
plt.show()